In [1]:
# ==========================================================
# Notebook 05: Hybrid Search and RRF Fusion
# ==========================================================

import os
import re
import pickle

import faiss
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df["search_text"] = enterprise_df["title"] + " " + enterprise_df["content"]

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team,search_text
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering,Phoenix Deployment Discussion The Phoenix-2026...
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance,Model Training Update The semantic search mode...
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR,Phoenix Architecture Wiki Project Phoenix-2026...
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering,Enterprise Search Design Hybrid search combine...
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering,PHX-245 Database Migration Complete PostgreSQL...


In [3]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)

    return text.split()

In [4]:
tokenized_corpus = [preprocess_text(doc) for doc in enterprise_df["search_text"]]

bm25 = BM25Okapi(tokenized_corpus)

In [5]:
bm25

In [6]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
document_embeddings = embedding_model.encode(
    enterprise_df["search_text"].tolist(), convert_to_numpy=True, show_progress_bar=True
)

document_embeddings = np.array(document_embeddings).astype("float32")

faiss.normalize_L2(document_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
embedding_dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(document_embeddings)

In [9]:
def bm25_retrieve(query, top_k=10):

    tokens = preprocess_text(query)

    scores = bm25.get_scores(tokens)

    ranking = np.argsort(scores)[::-1][:top_k]

    return list(ranking)

In [10]:
def semantic_retrieve(query, top_k=10):

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    return list(indices[0])

In [11]:
def reciprocal_rank_fusion(ranked_lists, k=60):

    rrf_scores = {}

    for ranked_list in ranked_lists:

        for rank, doc_id in enumerate(ranked_list, start=1):

            if doc_id not in rrf_scores:

                rrf_scores[doc_id] = 0.0

            rrf_scores[doc_id] += 1.0 / (k + rank)

    return rrf_scores

In [12]:
def hybrid_search(query, top_k=5):

    bm25_results = bm25_retrieve(query, top_k=20)

    semantic_results = semantic_retrieve(query, top_k=20)

    fused_scores = reciprocal_rank_fusion([bm25_results, semantic_results])

    ranked_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

    top_docs = ranked_docs[:top_k]

    output = []

    for doc_idx, score in top_docs:

        row = enterprise_df.iloc[doc_idx].to_dict()

        row["rrf_score"] = score

        output.append(row)

    return pd.DataFrame(output)

In [13]:
query = "PHX-245 vector database migration"

hybrid_results = hybrid_search(query, top_k=5)

hybrid_results[["doc_id", "title", "source", "rrf_score"]]

,doc_id,title,source,rrf_score
0,DOC0006,SEC-104 ACL Enhancement,Jira,0.191052
1,DOC0005,PHX-245 Database Migration,Jira,0.032787
2,DOC0004,Enterprise Search Design,Confluence,0.032002
3,DOC0001,Phoenix Deployment Discussion,Slack,0.032002
4,DOC0006,SEC-104 ACL Enhancement,Jira,0.030777


In [14]:
query = "PHX-245 vector database migration"

print("=" * 80)
print("BM25 TOP RESULTS")
print("=" * 80)

bm25_docs = bm25_retrieve(query, top_k=3)

print(enterprise_df.iloc[bm25_docs][["title"]])

print("\n")

print("=" * 80)
print("SEMANTIC TOP RESULTS")
print("=" * 80)

semantic_docs = semantic_retrieve(query, top_k=3)

print(enterprise_df.iloc[semantic_docs][["title"]])

print("\n")

print("=" * 80)
print("HYBRID RESULTS")
print("=" * 80)

print(hybrid_search(query, top_k=3)[["title", "rrf_score"]])

BM25 TOP RESULTS
                           title
4     PHX-245 Database Migration
3       Enterprise Search Design
0  Phoenix Deployment Discussion


SEMANTIC TOP RESULTS
                           title
4     PHX-245 Database Migration
0  Phoenix Deployment Discussion
3       Enterprise Search Design


HYBRID RESULTS
                        title  rrf_score
0     SEC-104 ACL Enhancement   0.191052
1  PHX-245 Database Migration   0.032787
2    Enterprise Search Design   0.032002


In [15]:
OUTPUT_PATH = "artifacts/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

with open(OUTPUT_PATH + "hybrid_search_config.pkl", "wb") as file:

    pickle.dump({"embedding_model": EMBEDDING_MODEL, "rrf_k": 60}, file)

print("Hybrid configuration saved!")

Hybrid configuration saved!
